# Load Data 

In [1]:
from torch.utils.data import DataLoader
from eb_jepa.datasets.moving_mnist import MovingMNISTDet


train_set = MovingMNISTDet(split="train")
val_set   = MovingMNISTDet(split="val")

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=2)

In [2]:
# batch = next(iter(train_loader))
# print(batch["video"].shape)
# print(batch["digit_location"].shape)

# Load model

In [3]:
import torch 
import torch.nn as nn 
from torch.optim import Adam
from tqdm import tqdm
from pathlib import Path 

from architectures import (
    ResNet5,
    ResUNet, 
    StateOnlyPredictor, 
    Projector,
    DetHead
    )
from image_decoder import ImageDecoder
from jepa import JEPA, JEPAProbe 
from losses import VCLoss, SquareLossSeq
from eval import validation_loop
from eb_jepa.training_utils import load_config, save_checkpoint

aritfact_dir = "artifacts/epoch_20.pth.tar"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
fname = "cfgs/default.yaml"
cfg = load_config(fname)

ckpt = torch.load(aritfact_dir, map_location= "cpu")

[INFO    ][2026-07-06 08:19:18][eb_jepa.training_utils][load_config              ] Loaded config from cfgs/default.yaml


In [4]:
encoder         = ResNet5(cfg.model.dobs, cfg.model.henc, cfg.model.dstc)
predictor_model = ResUNet(2 * cfg.model.dstc, cfg.model.hpre, cfg.model.dstc, is_rnn= False)    # is_rnn = False is default (being explicit).
predictor       = StateOnlyPredictor(predictor_model,context_length=2)
projector       = Projector(f"{cfg.model.dstc}-{cfg.model.dstc*4}-{cfg.model.dstc*4}")
regularizer     = VCLoss(cfg.loss.std_coeff, cfg.loss.cov_coeff, proj=projector)
ploss           = SquareLossSeq(projector)
jepa            = JEPA(encoder, encoder, predictor, regularizer, ploss).to(device)

# Initialize decoder and detection head (for evaluation only)
decoder        = ImageDecoder(cfg.model.dstc, cfg.model.dobs)
dethead        = DetHead(cfg.model.dstc, cfg.model.hpre, cfg.model.dobs)
pixel_decoder  = JEPAProbe(jepa, decoder, nn.MSELoss()).to(device)
detection_head = JEPAProbe(jepa, dethead, nn.BCELoss()).to(device)

encoder_params   = sum(p.numel() for p in encoder.parameters())
predictor_params = sum(p.numel() for p in predictor.parameters())
decoder_params   = sum(p.numel() for p in decoder.parameters())
dethead_params   = sum(p.numel() for p in dethead.parameters())

print(f'encoder_params :   {encoder_params}')
print(f'predictor_params : {predictor_params}')
print(f'decoder_params :   {decoder_params}')
print(f'dethead_params :   {dethead_params}')

encoder_params :   89280
predictor_params : 2032496
decoder_params :   2465
dethead_params :   4929


In [5]:
jepa.load_state_dict(ckpt["model_state_dict"])

<All keys matched successfully>

# Train DetHead, Decoder

In [6]:
jepa.eval() # prevents BatchNorm2d to updates 
detection_head.train()
pixel_decoder.train()

optimizer = Adam(
    [
        {"params": pixel_decoder.head.parameters(), "lr": cfg.optim.lr / 10},
        {"params": detection_head.head.parameters(), "lr": cfg.optim.lr}
    ]
)

In [7]:

path = "artifacts"
exp_dir = Path(path)
exp_dir.parent.mkdir(parents=True, exist_ok=True)

start_epoch = 0
global_step = 0 
recon_losses, det_losses = [], []

for epoch in range(start_epoch, cfg.optim.epochs):

    pbar = tqdm(
        train_loader,
        desc = f"Epoch {epoch}",
        disable=cfg.logging.get("tqdm_silent", False)
    )

    jepa.eval()

    for batch in pbar:
        batch = {k:v.to(device) for k, v in batch.items()}
        x = batch["video"]
        loc_map = batch["digit_location"]

        optimizer.zero_grad()

        recon_loss = pixel_decoder(x,x)
        det_loss = detection_head(x,loc_map)
        total_loss = recon_loss + det_loss



        total_loss.backward()
        optimizer.step()

        global_step += 1 

    print(f"train/recon_loss : {recon_loss.item()}")
    print(f"train/det_loss   : {det_loss.item()}")
    recon_losses.append(recon_loss.item())
    det_losses.append(det_loss.item())

    save_checkpoint(
            exp_dir / f"decoder_latest.pth.tar",
            model = pixel_decoder,
            optimizer=optimizer,
            epoch=epoch,
            step=global_step
        )
    
    save_checkpoint(
            exp_dir / f"detection_latest.pth.tar",
            model = detection_head ,
            optimizer=optimizer,
            epoch=epoch,
            step=global_step
        )


    if epoch % cfg.logging.save_every == 0 and epoch > 0:
        save_checkpoint(
            exp_dir / f"decoder_epoch_{epoch}.pth.tar",
            model = pixel_decoder,
            optimizer=optimizer,
            epoch=epoch,
            step=global_step
        )

        save_checkpoint(
            exp_dir / f"detection_epoch_{epoch}.pth.tar",
            model = detection_head ,
            optimizer=optimizer,
            epoch=epoch,
            step=global_step
        )
        


Epoch 0: 100%|██████████| 282/282 [02:30<00:00,  1.88it/s]

train/recon_loss : 0.017875051125884056
train/det_loss   : 0.04896680638194084
[INFO    ][2026-07-06 08:21:48][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/decoder_latest.pth.tar
[INFO    ][2026-07-06 08:21:49][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/detection_latest.pth.tar



Epoch 1: 100%|██████████| 282/282 [02:55<00:00,  1.61it/s]

train/recon_loss : 0.011916622519493103
train/det_loss   : 0.04620379954576492
[INFO    ][2026-07-06 08:24:44][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/decoder_latest.pth.tar
[INFO    ][2026-07-06 08:24:44][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/detection_latest.pth.tar



Epoch 2: 100%|██████████| 282/282 [03:09<00:00,  1.49it/s]

...


Epoch 48: 100%|██████████| 282/282 [03:51<00:00,  1.22it/s]

train/recon_loss : 0.003363931318745017
train/det_loss   : 0.028206676244735718
[INFO    ][2026-07-06 11:14:57][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/decoder_latest.pth.tar
[INFO    ][2026-07-06 11:14:57][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/detection_latest.pth.tar



Epoch 49: 100%|██████████| 282/282 [03:59<00:00,  1.18it/s]

train/recon_loss : 0.003625542391091585
train/det_loss   : 0.04406270012259483
[INFO    ][2026-07-06 11:18:57][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/decoder_latest.pth.tar


[INFO    ][2026-07-06 11:18:57][eb_jepa.training_utils][save_checkpoint          ] Saved checkpoint: artifacts/detection_latest.pth.tar


In [8]:
recon_losses

[0.017875051125884056,
 0.011916622519493103,
 0.010520832613110542,
 0.009642671793699265,
 0.00820162147283554,
 0.008164308965206146,
 0.00742764538154006,
 0.008314618840813637,
 0.007495907135307789,
 0.007548472378402948,
 0.007455923594534397,
 0.006303758360445499,
 0.006174908019602299,
 0.00594155490398407,
 0.0058651999570429325,
 0.006108131259679794,
 0.005024425219744444,
 0.005673889070749283,
 0.005520958453416824,
 0.005458374973386526,
 0.005328529980033636,
 0.004589914809912443,
 0.004605400841683149,
 0.004734245594590902,
 0.004734371323138475,
 0.0044944919645786285,
 0.00432064663618803,
 0.004219047725200653,
 0.004536285996437073,
 0.004437303636223078,
 0.003986950498074293,
 0.003980423789471388,
 0.0037490117829293013,
 0.0041659832932055,
 0.0039051224011927843,
 0.003593277186155319,
 0.003638832364231348,
 0.003765637753531337,
 0.0035778977908194065,
 0.003535965457558632,
 0.003670361591503024,
 0.003859670599922538,
 0.0035307027865201235,
 0.00376930

In [10]:
[0.017875051125884056,
 0.011916622519493103,
 0.010520832613110542,
 0.009642671793699265,
 0.00820162147283554,
 0.008164308965206146,
 0.00742764538154006,
 0.008314618840813637,
 0.007495907135307789,
 0.007548472378402948,
 0.007455923594534397,
 0.006303758360445499,
 0.006174908019602299,
 0.00594155490398407,
 0.0058651999570429325,
 0.006108131259679794,
 0.005024425219744444,
 0.005673889070749283,
 0.005520958453416824,
 0.005458374973386526,
 0.005328529980033636,
 0.004589914809912443,
 0.004605400841683149,
 0.004734245594590902,
 0.004734371323138475,
 0.0044944919645786285,
 0.00432064663618803,
 0.004219047725200653,
 0.004536285996437073,
 0.004437303636223078,
 0.003986950498074293,
 0.003980423789471388,
 0.0037490117829293013,
 0.0041659832932055,
 0.0039051224011927843,
 0.003593277186155319,
 0.003638832364231348,
 0.003765637753531337,
 0.0035778977908194065,
 0.003535965457558632,
 0.003670361591503024,
 0.003859670599922538,
 0.0035307027865201235,
 0.0037693025078624487,
 0.0037875839043408632,
 0.0037784017622470856,
 0.0033607028890401125,
 0.003384349634870887,
 0.003363931318745017,
 0.003625542391091585]

[0.017875051125884056,
 0.011916622519493103,
 0.010520832613110542,
 0.009642671793699265,
 0.00820162147283554,
 0.008164308965206146,
 0.00742764538154006,
 0.008314618840813637,
 0.007495907135307789,
 0.007548472378402948,
 0.007455923594534397,
 0.006303758360445499,
 0.006174908019602299,
 0.00594155490398407,
 0.0058651999570429325,
 0.006108131259679794,
 0.005024425219744444,
 0.005673889070749283,
 0.005520958453416824,
 0.005458374973386526,
 0.005328529980033636,
 0.004589914809912443,
 0.004605400841683149,
 0.004734245594590902,
 0.004734371323138475,
 0.0044944919645786285,
 0.00432064663618803,
 0.004219047725200653,
 0.004536285996437073,
 0.004437303636223078,
 0.003986950498074293,
 0.003980423789471388,
 0.0037490117829293013,
 0.0041659832932055,
 0.0039051224011927843,
 0.003593277186155319,
 0.003638832364231348,
 0.003765637753531337,
 0.0035778977908194065,
 0.003535965457558632,
 0.003670361591503024,
 0.003859670599922538,
 0.0035307027865201235,
 0.00376930

In [9]:
det_losses

[0.04896680638194084,
 0.04620379954576492,
 0.043056536465883255,
 0.038376301527023315,
 0.03495408967137337,
 0.038702793419361115,
 0.047056522220373154,
 0.04275905713438988,
 0.03922318294644356,
 0.04283842444419861,
 0.03267785534262657,
 0.03563767299056053,
 0.03741547465324402,
 0.03221551701426506,
 0.047754984349012375,
 0.041948914527893066,
 0.029923269525170326,
 0.045538775622844696,
 0.029783857986330986,
 0.040782470256090164,
 0.030378593131899834,
 0.04352322220802307,
 0.026086444035172462,
 0.029142189770936966,
 0.031031448394060135,
 0.0476287305355072,
 0.03194437548518181,
 0.024795150384306908,
 0.031972676515579224,
 0.02795102633535862,
 0.030834585428237915,
 0.033456623554229736,
 0.027208993211388588,
 0.028508229181170464,
 0.03638886660337448,
 0.032408457249403,
 0.031379107385873795,
 0.04035719856619835,
 0.027054691687226295,
 0.027062876150012016,
 0.033549197018146515,
 0.026231426745653152,
 0.03168734908103943,
 0.04065198078751564,
 0.0285649

In [11]:
[0.04896680638194084,
 0.04620379954576492,
 0.043056536465883255,
 0.038376301527023315,
 0.03495408967137337,
 0.038702793419361115,
 0.047056522220373154,
 0.04275905713438988,
 0.03922318294644356,
 0.04283842444419861,
 0.03267785534262657,
 0.03563767299056053,
 0.03741547465324402,
 0.03221551701426506,
 0.047754984349012375,
 0.041948914527893066,
 0.029923269525170326,
 0.045538775622844696,
 0.029783857986330986,
 0.040782470256090164,
 0.030378593131899834,
 0.04352322220802307,
 0.026086444035172462,
 0.029142189770936966,
 0.031031448394060135,
 0.0476287305355072,
 0.03194437548518181,
 0.024795150384306908,
 0.031972676515579224,
 0.02795102633535862,
 0.030834585428237915,
 0.033456623554229736,
 0.027208993211388588,
 0.028508229181170464,
 0.03638886660337448,
 0.032408457249403,
 0.031379107385873795,
 0.04035719856619835,
 0.027054691687226295,
 0.027062876150012016,
 0.033549197018146515,
 0.026231426745653152,
 0.03168734908103943,
 0.04065198078751564,
 0.02856498397886753,
 0.031072644516825676,
 0.03143847733736038,
 0.0339013896882534,
 0.028206676244735718,
 0.04406270012259483]

[0.04896680638194084,
 0.04620379954576492,
 0.043056536465883255,
 0.038376301527023315,
 0.03495408967137337,
 0.038702793419361115,
 0.047056522220373154,
 0.04275905713438988,
 0.03922318294644356,
 0.04283842444419861,
 0.03267785534262657,
 0.03563767299056053,
 0.03741547465324402,
 0.03221551701426506,
 0.047754984349012375,
 0.041948914527893066,
 0.029923269525170326,
 0.045538775622844696,
 0.029783857986330986,
 0.040782470256090164,
 0.030378593131899834,
 0.04352322220802307,
 0.026086444035172462,
 0.029142189770936966,
 0.031031448394060135,
 0.0476287305355072,
 0.03194437548518181,
 0.024795150384306908,
 0.031972676515579224,
 0.02795102633535862,
 0.030834585428237915,
 0.033456623554229736,
 0.027208993211388588,
 0.028508229181170464,
 0.03638886660337448,
 0.032408457249403,
 0.031379107385873795,
 0.04035719856619835,
 0.027054691687226295,
 0.027062876150012016,
 0.033549197018146515,
 0.026231426745653152,
 0.03168734908103943,
 0.04065198078751564,
 0.0285649